# Context Managers and the `with` Statement

## 🎯 Learning Objectives
- Understand what a Context Manager is and why it is critical for Resource Management.
- Master the `with` statement for File Handling.
- Understand how Context Managers prevent memory leaks and file lock errors.
- Learn how to create your own custom Context Manager (Advanced).

## 📝 Prerequisites
- [01-Read-Write-Files](01-Read-Write-Files.ipynb)
- [01-Classes-and-Objects](../06-OOP/01-Classes-and-Objects.ipynb)

## 📖 Theory & Concepts

When your code interacts with the "outside world" (like opening a file, connecting to a database, or opening a network socket), you are taking control of a **System Resource**.

If your code crashes while holding that resource, the resource remains "locked" and can cause memory leaks or prevent other programs from accessing the file.

A **Context Manager** is a Python construct (used via the `with` keyword) that guarantees a resource will be properly cleaned up and closed, **even if an error occurs** during execution.

---
## 💻 1. The "Old" Way vs The "Pythonic" Way

Before Context Managers, you had to manually use `try-finally` blocks to guarantee a file was closed.


In [ ]:
# ❌ THE OLD, DANGEROUS WAY
file = open("test.txt", "w")
try:
    file.write("Hello World")
    # If an exception happened here, it skips the rest of the try block
finally:
    file.close() # This guarantees the file closes

# ✅ THE PYTHONIC WAY (Using a Context Manager)
with open("test.txt", "w") as file:
    file.write("Hello World")
# The file is AUTOMATICALLY closed here, even if it crashed on the line above!

---
## 💻 2. Creating Custom Context Managers (OOP)

Under the hood, the `with` statement looks for two "Magic Methods" inside a class:
- `__enter__(self)`: What happens when we OPEN the context.
- `__exit__(self, exc_type, exc_val, traceback)`: What happens when we CLOSE the context.


In [ ]:
class CustomFileOpener:
    def __init__(self, filename, mode):
        self.filename = filename
        self.mode = mode
        self.file = None

    def __enter__(self):
        print(f"[Enter] Opening {self.filename}")
        self.file = open(self.filename, self.mode)
        return self.file # This is what gets assigned to the 'as' variable

    def __exit__(self, exc_type, exc_val, traceback):
        print(f"[Exit] Closing {self.filename}")
        if self.file:
            self.file.close()
        # If there was an error, exc_type would hold it here. 
        # Returning True suppresses the error. Returning None raises it.

# Using our custom Context Manager
with CustomFileOpener("test.txt", "w") as f:
    f.write("Testing custom context manager!")
    print("Writing to file...")

---
## 💻 3. Creating Context Managers using `@contextmanager`
Instead of writing a full Class, you can use the `contextlib` library to turn a Generator (using `yield`) into a Context Manager!


In [ ]:
from contextlib import contextmanager

@contextmanager
def simple_file_opener(filename, mode):
    print(f"[Enter] Opening {filename}")
    file = open(filename, mode)
    try:
        yield file # Code inside the 'with' block runs here!
    finally:
        print(f"[Exit] Closing {filename}")
        file.close()

with simple_file_opener("test.txt", "w") as f:
    f.write("Using contextlib!")

---
## 🌍 Real-world Example (Timing Execution)
Context Managers aren't just for files! You can use them to time how long a block of code takes to execute.

In [ ]:
import time
from contextlib import contextmanager

@contextmanager
def timer(block_name: str):
    start_time = time.time()
    yield # Let the code inside the block run
    end_time = time.time()
    print(f"{block_name} took {end_time - start_time:.5f} seconds")

# Using the timer
with timer("Big Loop"):
    x = [i**2 for i in range(1_000_000)]

---
## 🛠️ Practice Problems

### Problem 1: Suppress Exceptions (Medium)
**Description:** Write a custom class-based context manager called `SuppressErrors`. If a `ZeroDivisionError` occurs inside the `with` block, the program should NOT crash. Instead, it should just print "Error Suppressed" and continue. If any other error occurs, it should crash normally.

In [ ]:
class SuppressErrors:
    # TODO: Implement __enter__ and __exit__
    pass

# --- TEST CASES ---
# Run this cell to test your code!
# with SuppressErrors():
#     print("About to divide by zero...")
#     x = 1 / 0
#     print("This won't print")
# print("✅ The program didn't crash! Problem 1 passed!")

---
## 🧠 Interview Questions
1. **What happens if you don't close a file?**
   *Answer: It creates a memory leak, locks the file from being edited by other processes, and data might not be completely flushed (written) to the disk. Always use `with open(...)`!*
2. **What two magic methods must be implemented to create a custom context manager class?**
   *Answer: `__enter__` and `__exit__`.*

---
## ⚠️ Common Mistakes
| Bad Code | Good Code (Pythonic) |
|----------|----------------------|
| `f = open('data.csv')`<br>`data = f.read()`<br>`f.close()` | `with open('data.csv') as f:`<br>&nbsp;&nbsp;&nbsp;&nbsp;`data = f.read()` |

---
## 🔑 Key Takeaways
- The `with` statement ensures resources are cleaned up automatically.
- You should **never** use `open()` without `with`.
- You can build custom context managers using classes (`__enter__`/`__exit__`) or generators (`@contextmanager`).
